---
title: Building A Seismic Design
project: Beaverton Creak Apartments
location: Beaverton, OR
client: GBD Architects
job_number: 10022500446
author:
  name: Andy Sazima
  initials: AGS
  email: andy.sazima@kpff.com
---

In [ ]:
%reset -f
import math
import os
import warnings
from collections.abc import Callable
from math import pow, sqrt

import forallpeople as units
import handcalcs
import handcalcs.render
import numpy as np

import pandas as pd
import sympy as sp
from structural_tools.notebook.display import display_table, display_text, sig_figs
from structural_tools.notebook.calculation import (
    check_value,
    set_params_columns,
)
from structural_tools.seismic.asce.parameters import LOAD_FACTOR_SEISMIC_ASD, RiskCategory, SeismicParameters, SiteClass

units.environment(env_name="structural", top_level=True)
handcalcs.set_option("custom_symbols", {"star": "^*", "__": ",", "plus": "+", "minus": "-"})
handcalcs.set_option("greek_exclusions", ["psi"])
handcalcs.set_option("display_precision", 4)
handcalcs.set_option("param_columns", 1)
handcalcs.set_option("latex_block_start", "\\begin{equation*}")
handcalcs.set_option("latex_block_end", "\\end{equation*}")

# Effective Seismic Weight Calculation

We calculate the effective seismic weight with unit weights and square footages.

## Unit Weights

Weights will be assigned with area unit weights.

In [ ]:
set_params_columns(4)

In [ ]:
%%render params
U_corridor = 35 / 1000 * ksf
U_floor = 35 / 1000 * ksf
U_roof = 25 / 1000 * ksf
U_wall = 12 / 1000 * ksf

## Building Geometry

The lengths and widths of the weight areas are consistent across levels 2, 3, and 4.

In [ ]:
set_params_columns(5)

In [ ]:
%%render params
L_floor = 230 * ft
w_floor = (30 + 9 / 12) * ft
L_corridor = 230 * ft
w_corridor = 5.5 * ft
w_building = (w_floor * 2) + w_corridor

The wall heights vary across the levels.

In [ ]:
%%render params
h_wall__1 = 11.5 * ft
h_wall__2 = 10.5 * ft
h_wall__3 = 10.5 * ft
h_wall__4 = 10 * ft

There is a small jog in the building floorplan. This does not affect the area, but the perimeter will be longer.

In [ ]:
%%render params
L_jog = 5 * ft

In [ ]:
%%render
P_wall = (2 * L_floor) + (2 * w_building) + (2 * L_jog)

Areas can now be calculated. Note that the areas of the walls are the actual wall areas; they have not been distributed to each floor based on tributary area, yet.

In [ ]:
%%render 
A_floor = 2 * (L_floor * w_floor)
# This is a slight overestimate as the corridor doesn't stretch the full length of the building
A_corridor = L_corridor * w_corridor
A_level = L_floor * w_building
A_wall__1 = h_wall__1 * P_wall
A_wall__2 = h_wall__2 * P_wall
A_wall__3 = h_wall__3 * P_wall
A_wall__4 = h_wall__4 * P_wall

## Component Weights

The typical floor and corridor weights are

In [ ]:
%%render
W_floor__typ = U_floor * A_floor
W_corridor__typ = U_corridor * A_corridor

This applies to levels 2, 3, and 4.

In [ ]:
set_params_columns(3)

In [ ]:
%%render params
W_floor__2 = W_floor__typ
W_floor__3 = W_floor__typ
W_floor__4 = W_floor__typ
W_corridor__2 = W_corridor__typ
W_corridor__3 = W_corridor__typ
W_corridor__4 = W_corridor__typ

The roof weight is

In [ ]:
%%render
W_roof = U_roof * A_level

The exterior walls are divided in half to the floor above and below.

In [ ]:
%%render
W_wall__2 = U_wall * (A_wall__2 / 2 + A_wall__1 / 2)
W_wall__3 = U_wall * (A_wall__3 / 2 + A_wall__2 / 2)
W_wall__4 = U_wall * (A_wall__4 / 2 + A_wall__3 / 2)
W_wall__roof = U_wall * (A_wall__4 / 2)

## Level Weights

Each level's weight is

In [ ]:
%%render 
W_level__2 = W_floor__2 + W_corridor__2 + W_wall__2
W_level__3 = W_floor__3 + W_corridor__3 + W_wall__3
W_level__4 = W_floor__4 + W_corridor__4 + W_wall__4
# be sure to add the weight of the truss later
W_level__roof = W_roof + W_wall__roof

## Total Weight

The total weight of the structure is

In [ ]:
%%render long
W_total = W_level__2 + W_level__3 + W_level__4 + W_level__roof

# Seismic Equivalent Lateral Force (ELF) Analysis

We will be using the equivalent lateral force method prescribed by ASCE 7-16 Section 12.8. We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

## Seismic Parameters

In [ ]:
S_S = 0.887  # g
S_1 = 0.410  # g
seismic_params = SeismicParameters(s_1=S_1, s_s=S_S, risk_category=RiskCategory.II, site_class=SiteClass.D)

Seismic design parameters are obtained from the geotech report and ASCE 7-16 Sec. 11 and 12.

In [ ]:
set_params_columns(5)

In [ ]:
%%render params
R = 6.5
Omega_0 = 3.0
C_d = 4.0
rho = 1.0

In [ ]:
%%render params
I_e = seismic_params.i_e
S_S = seismic_params.s_s  # g
S_1 = seismic_params.s_1  # g
F_a = seismic_params.f_a
F_v = seismic_params.f_v
S_MS = seismic_params.s_ms  # g
S_M1 = seismic_params.s_m1  # g
S_DS = seismic_params.s_ds  # g
S_D1 = seismic_params.s_d1  # g
SDC = seismic_params.sdc
C_u = 1.4
T_L = 16  # s

## Base Shear

We must calculate the period

In [ ]:
set_params_columns(1)

In [ ]:
%%render params
h_top__roof = (58 + 3 / 12) * ft
h_bottom__roof = 42.5 * ft
h_n = np.mean([h_top__roof, h_bottom__roof])  # ASCE 7-16 Sec. 11.2

In [ ]:
%%render
h_n__check = check_value(h_n, 65 * ft, "leq")

We will approximate the building period.

In [ ]:
h_n = float(h_n)  # to prevent weirdness with units

In [ ]:
%%render
C_t = 0.02  # ASCE 7-16 Table 12.8-2
x = 0.75  # ASCE 7-16 Table 12.8-2
T_a = C_t * h_n**x  # ASCE 7-16 Eq. 12.8-7

Because we are not using an automated analysis software, we can make the following assumption:

In [ ]:
%%render
T = T_a  # ASCE 7-16 Sec. 12.8.2

With the period, we can calculate the seismic response coefficient.

In [ ]:
%%render
C_s = S_DS / (R / I_e)  # ASCE 7-16 Eq. 12.8-2

We must check that $C_s$ does not exceed:

In [ ]:
%%render long
# fmt: off
if T <= T_L:  C_s__check__max = S_D1 / (T * (R / I_e));              C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
elif T > T_L: C_s__check__max = (S_D1 * T_L) / (T**2 * (R / I_e));   C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
# fmt: on

And we must also check that $C_s$ is not less than:

In [ ]:
%%render
C_s__check__min__1 = 0.044 * S_DS * I_e  # ASCE 7-16 Eq. 12.8-5
C_s__check__min = check_value(C_s, C_s__check__min__1, "geq")

as well as:

In [ ]:
%%render
C_s__check__min = check_value(C_s, 0.01, "geq")  # ASCE 7-16 Eq. 12.8-5

and finally:

In [ ]:
%%render
# fmt: off
if S_1 >= 0.6:  C_s__check__min__3 = 0.5 * S_1 / (R / I_e); C_s__check__min = check_value(C_s, C_s__check__min__3, "geq")  # ASCE 7-16 Eq. 12.8-6
elif S_1 < 0.6: C_s__check__min = "\\text{skipping check}"  # ASCE 7-16 Eq. 12.8-6
# fmt: on

And now, we can calculate the base shear, $V$

In [ ]:
%%render
V = C_s * W_total  # ASCE 7-16 Eq. 12.8-1

## Distribute the seismic forces

With the base shear calculated, we must now distribute the base shear on each level.

First, we must calculate the structural period exponent, $k$.

In [ ]:
%%render
# fmt: off
if T <= 0.5: k = 1
elif T >= 2.5: k = 2
elif T > 0.5 and T < 2.5: k = 1 + ((T - 0.5) * (2 - 1)) / (2.5 - 0.5)
# fmt: on

Now, we can distribute the forces per floor,

$$
F_x = C_{vx} V\;\;\text{(ASCE 7-16 Eq. 12.8-11)}
$$

As well as cumulatively from roof to ground.

$$
V_x = \sum_{i=x}^n F_i\;\; \text{(ASCE 7-16 Eq. 12.8-11)}
$$

In [ ]:
# Load in basic data
seismic_loads = pd.DataFrame(
    {
        "Level": ["Level 1", "Level 2", "Level 3", "Level 4", "Roof"],
        "floor height": [
            float(h_wall__1),
            float(h_wall__2),
            float(h_wall__3),
            float(h_wall__4),
            0,
        ],
        "floor weight": [
            0,
            float(W_level__2),
            float(W_level__3),
            float(W_level__4),
            float(W_level__roof),
        ],
    },
)
seismic_loads = seismic_loads.set_index("Level")

# Calc cumulative elevation
elevations = [0]
for height in seismic_loads["floor height"][:-1]:
    elevations.append(height + elevations[-1])
seismic_loads.insert(2, "elevation", elevations)

# Calc coefficients and story forces ASCE 7-16 Eq. 12.8-11 and 12.8-12
seismic_loads["wxhxk"] = seismic_loads["floor weight"] * seismic_loads["elevation"] ** float(k)
cumulative_wxhxk = seismic_loads["wxhxk"].sum()
seismic_loads["C_vx"] = seismic_loads["wxhxk"] / cumulative_wxhxk
seismic_loads["F_x"] = seismic_loads["C_vx"] * float(V)

# Calc story shear forces ASCE 7-16 Eq. 12.8-13
story_shears = [0]
for shear in reversed(list(seismic_loads["F_x"][1:])):
    story_shears.insert(0, shear + story_shears[0])
seismic_loads["V_x"] = story_shears

# Calc overturning moments
floor_moments = seismic_loads["V_x"] * seismic_loads["floor height"]
overturning_moments = [0]
for floor_moment in reversed(list(floor_moments[:-1])):
    overturning_moments.insert(0, overturning_moments[0] + floor_moment)
seismic_loads["OTM"] = overturning_moments

In [ ]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "floor height": "$h_{floor}$ [ft]",
    "floor weight": "$w_x$ [kip]",
    "elevation": "$h_x$ [ft]",
    "wxhxk": "$w_x h_x^k$",
    "C_vx": "$C_{vx}$",
    "F_x": "$F_x$ [kip]",
    "V_x": "$V_x$ [kip]",
    "OTM": "OTM [kip-ft]",
}

Calculating for each floor leads to the following \autoref{tab:seismic_forces_calculation_table}

In [ ]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
display_table(
    dataframe=seismic_loads.iloc[::-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Seismic Forces Calculation Table",
    label="tab:seismic_forces_calculation_table",
)

# Diaphragm and Shear Wall Design

## Redundancy Factor

The redundancy factor, $\rho$, is either 1.3 or 1.0 according to ASCE 7-16 12.3.4.

No single story resists more than 35% of the shear force (see $C_{vx}$ in \autoref{tab:seismic_forces_calculation_table}), so all values of $\rho$ are $1.0$.

## Shear Wall Design

We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

In [ ]:
# Set up diaphragm dataframe
diaphragm_rigidity = ["Flexible", "Rigid"]
levels = ["Level 4", "Level 3", "Level 2", "Level 1"]
walls = {
    "A-2": "N-S",
    "A-2.4": "N-S",
    "A-3": "N-S",
    "A-3.7": "N-S",
    "A-4": "N-S",
    "A-5": "N-S",
    "A-5.1": "N-S",
    "A-6": "N-S",
    "A-6.1": "N-S",
    "A-6.8": "N-S",
    "A-7": "N-S",
    "A-7.8": "N-S",
    "A-8": "N-S",
    "A-8.2": "N-S",
    "A-8.9": "N-S",
    "A-9": "N-S",
    # E-W
    "A-B--A-1": "E-W",
    "A-B--A-2.5": "E-W",
    "A-B--A-3.5": "E-W",
    "A-B--A-4": "E-W",
    "A-B--A-4.8": "E-W",
    "A-B--A-5.5": "E-W",
    "A-B--A-6.5": "E-W",
    "A-B--A-7.9": "E-W",
    "A-B--A-8.8": "E-W",
    "A-C--A-1.8": "E-W",
    "A-C--A-3.2": "E-W",
    "A-C--A-3.8": "E-W",
    "A-C--A-4.6": "E-W",
    "A-C--A-5.5": "E-W",
    "A-C--A-6.5": "E-W",
    "A-C--A-6.9": "E-W",
    "A-C--A-8.5": "E-W",
    "A-C--A-9.2": "E-W",
}

index = pd.MultiIndex.from_product(
    [levels, walls.keys()],
    names=["Level", "Wall"],
)

shear_walls = pd.DataFrame(index=index)
shear_walls["Direction"] = shear_walls.index.get_level_values("Wall").map(walls)
flexible = shear_walls.copy()
flexible_ns = flexible[flexible["Direction"] == "N-S"].copy()
flexible_ew = flexible[flexible["Direction"] == "E-W"].copy()
rigid = shear_walls.copy()
rigid_ns = rigid[rigid["Direction"] == "N-S"].copy()
rigid_ew = rigid[rigid["Direction"] == "E-W"].copy()

### Flexible Diaphragm Assumption

We will uniformly distribute the shear forces across the diaphragms based on tributary areas of each level. This removes some nuance in the distribution of floor weight distribution (corridors are 17% heavier than the rest of the floors, for instance), but the vast majority of the area of each level is typical floor weight (30 psf) instead of corridor weight (35 psf).

$$
f_{p,x} = F_x / A_{level}
$$

In [ ]:
%%render
A_level
f_p__roof = seismic_loads.loc["Roof", "F_x"] * 1000 * lb / A_level
f_p__4 = seismic_loads.loc["Level 4", "F_x"] * 1000 * lb / A_level
f_p__3 = seismic_loads.loc["Level 3", "F_x"] * 1000 * lb / A_level
f_p__2 = seismic_loads.loc["Level 2", "F_x"] * 1000 * lb / A_level

#### North-South Direction

Most of the shear walls are not in line which presents an issue: if we assume a diaphragm is the full depth, there will be an extremely long collector element without a shear wall on both sides. We will avoid this by just assuming the flexible diaphragm does not extend the full depth and instead is just tributary to the nearest wall (\autoref{fig:diaphragm_trib_area}). This means that there are essentially two diaphragms through the depth of the building. This is a rough approximation to ensure all the shear load is resolved in the shear walls. \

The walls that are close to inline (A-5 and A-5.1, A-6 and A-6.1, and A-9 and A-9.1) in reality will be treated as inline and have collectors to transfer the shear load from the diaphragm to those walls.

Proper detailing will be done to ensure continuity through the depth of the diaphragm.

![Shear walls (thick red lines) in plan view with their (colored) tributary areas.](images/shear_wall_tributary_area.png){width=60% #fig:diaphragm_trib_area}

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ns} through \autoref{tab:shear_wall_forces_level2_ns}.

Shear wall sheathing will be assumed to have the following properties:

1. WSP Sheathing
2. 15/32" thickness
3. 10d common nails
4. Plywood panels
5. Framing uses Douglas-Fir-Larch (no specific gravity adjustment)

Shear capacity and stiffness values come from **AWC SDPWS 2021 Table 4.3A**.

Notes: 

- Cumulative shear force, $F_{x,cum}$, is cumulative from the Roof downwards. For example, Level 3 includes the shear forces from the Roof, Level 4, and Level 3.

In [ ]:
from structural_tools.seismic.sheathing import (
    DesignMethod,
    LoadCase,
    Nail,
    PanelType,
    Sheathing,
    SheathingMaterial,
    get_sheathing_properties,
)

# Set baseline sheathing properties. We set the number of sheathed sides later
sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
)

In [ ]:
# Plan dimensions
l_A1__A2 = (31 + 11 / 12) * ft
l_A2__A3 = (17 + 11 / 12) * ft
l_A3__A4 = (26 + 4 / 12) * ft
l_A4__A5 = (21 + 4 / 12) * ft
l_A5__A6 = (24 + 8 / 12) * ft
l_A6__A7 = (34 + 4 / 12) * ft
l_A7__A8 = (26 + 1 / 12) * ft
l_A8__A9 = (21 + 4 / 12) * ft
l_A9__A10 = (31 + 1 / 12) * ft

l_A1__A24 = (40 + 1 / 12) * ft
l_A24__A35 = (23 + 8 / 12) * ft
l_A35__A51 = (34 + 4 / 12) * ft
l_A51__A61 = (24 + 8 / 12) * ft
l_A61__A68 = (24 + 8 / 12) * ft
l_A68__A82 = (38 + 3 / 12) * ft
l_A82__A89 = (17 + 11 / 12) * ft
l_A89__A10 = (31 + 5 / 12) * ft

l_A37__A4 = (8 + 10 / 12) * ft
l_A3__A37 = l_A3__A4 - l_A37__A4
l_A78__A82 = (8 + 10 / 12) * ft
l_A68__A78 = l_A68__A82 - l_A78__A82

d_trib_ns = 67 / 2 * ft

In [ ]:
# Wall tributary widths and lengths
t_w__A2 = l_A1__A2 + (l_A2__A3) / 2
t_w__A24 = (l_A1__A24 - L_jog) + ((l_A24__A35 + l_A35__A51) / 2)
t_w__A3 = (l_A2__A3 / 2) + (l_A3__A37 / 2)
t_w__A37 = (l_A3__A37 / 2) + (l_A37__A4 / 2)
t_w__A4 = (l_A37__A4 / 2) + (l_A4__A5 / 2)
t_w__A5 = (l_A4__A5 / 2) + (l_A5__A6 / 2)
t_w__A51 = ((l_A24__A35 + l_A35__A51) / 2) + (l_A51__A61 / 2)
t_w__A6 = (l_A5__A6 / 2) + (l_A6__A7 / 2)
t_w__A61 = (l_A51__A61 / 2) + (l_A61__A68 / 2)
t_w__A68 = (l_A61__A68 / 2) + (l_A68__A78 / 2)
t_w__A7 = (l_A6__A7 / 2) + (l_A7__A8 / 2)
t_w__A78 = (l_A68__A78 / 2) + (l_A78__A82 / 2)
t_w__A8 = (l_A7__A8 / 2) + (l_A8__A9 / 2)
t_w__A82 = (l_A78__A82 / 2) + (l_A82__A89 / 2)
t_w__A89 = (l_A82__A89 / 2) + l_A89__A10
t_w__A9 = (l_A8__A9 / 2) + (l_A9__A10 - L_jog)

t_l__A2 = float(d_trib_ns)
t_l__A24 = float(d_trib_ns)
t_l__A3 = float(d_trib_ns)
t_l__A37 = float(d_trib_ns)
t_l__A4 = float(d_trib_ns)
t_l__A5 = float(d_trib_ns)
t_l__A51 = float(d_trib_ns)
t_l__A6 = float(d_trib_ns)
t_l__A61 = float(d_trib_ns)
t_l__A68 = float(d_trib_ns)
t_l__A7 = float(d_trib_ns)
t_l__A78 = float(d_trib_ns)
t_l__A8 = float(d_trib_ns)
t_l__A82 = float(d_trib_ns)
t_l__A89 = float(d_trib_ns)
t_l__A9 = float(d_trib_ns)

In [ ]:
flexible_ns.loc["Level 4", "tributary weight"] = float(f_p__roof)
flexible_ns.loc["Level 3", "tributary weight"] = float(f_p__4)
flexible_ns.loc["Level 2", "tributary weight"] = float(f_p__3)
flexible_ns.loc["Level 1", "tributary weight"] = float(f_p__2)

# Shear wall height is for the floor below
flexible_ns.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ns.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ns.loc["Level 2", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ns.loc["Level 1", "wall height"] = float(seismic_loads.loc["Level 1", "floor height"])

# Add wall tributary widths and wall lengths to df
l_aa__ab = 29 + 9 / 12
l_ac__ad = 29 + 9 / 12
l_stairwall = 20
wall_tributary = {
    "A-2": {"trib_width": float(t_w__A2), "trib_depth": t_l__A2, "wall_length": l_aa__ab},
    "A-2.4": {"trib_width": float(t_w__A24), "trib_depth": t_l__A24, "wall_length": l_ac__ad},
    "A-3": {"trib_width": float(t_w__A3), "trib_depth": t_l__A37, "wall_length": l_aa__ab},
    "A-3.7": {"trib_width": float(t_w__A37), "trib_depth": t_l__A37, "wall_length": l_stairwall},
    "A-4": {"trib_width": float(t_w__A4), "trib_depth": t_l__A4, "wall_length": l_stairwall},
    "A-5": {"trib_width": float(t_w__A5), "trib_depth": t_l__A5, "wall_length": l_aa__ab},
    "A-5.1": {"trib_width": float(t_w__A51), "trib_depth": t_l__A51, "wall_length": l_ac__ad},
    "A-6": {"trib_width": float(t_w__A6), "trib_depth": t_l__A6, "wall_length": l_aa__ab},
    "A-6.1": {"trib_width": float(t_w__A61), "trib_depth": t_l__A61, "wall_length": l_ac__ad},
    "A-6.8": {"trib_width": float(t_w__A68), "trib_depth": t_l__A68, "wall_length": l_ac__ad},
    "A-7": {"trib_width": float(t_w__A7), "trib_depth": t_l__A7, "wall_length": l_aa__ab},
    "A-7.8": {"trib_width": float(t_w__A78), "trib_depth": t_l__A78, "wall_length": l_stairwall},
    "A-8": {"trib_width": float(t_w__A8), "trib_depth": t_l__A8, "wall_length": l_aa__ab},
    "A-8.2": {"trib_width": float(t_w__A82), "trib_depth": t_l__A82, "wall_length": l_stairwall},
    "A-8.9": {"trib_width": float(t_w__A89), "trib_depth": t_l__A89, "wall_length": l_ac__ad},
    "A-9": {"trib_width": float(t_w__A9), "trib_depth": t_l__A9, "wall_length": l_aa__ab},
}

flexible_ns["wall length"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary width"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary depth"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary area"] = flexible_ns["tributary depth"] * flexible_ns["tributary width"]

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ns["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up. Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ns["line"] = flexible_ns.index.get_level_values("Wall").str.split("--").str[0]
# Get tributary area and wall length for the shear line
flexible_ns["line tributary area"] = flexible_ns.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ns["line wall length"] = flexible_ns.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ns["line shear demand"] = flexible_ns["line tributary area"] * flexible_ns["tributary weight"]
flexible_ns["line unit shear"] = flexible_ns["line shear demand"] / flexible_ns["line wall length"]
flexible_ns["line cumulative shear demand"] = flexible_ns.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ns["line cumulative unit shear"] = (
    flexible_ns["line cumulative shear demand"] / flexible_ns["line wall length"]
)

flexible_ns["shear demand"] = flexible_ns["line unit shear"] * flexible_ns["wall length"]
flexible_ns["cumulative shear demand"] = flexible_ns.groupby(level="Wall")["shear demand"].cumsum()
flexible_ns["unit shear demand"] = flexible_ns["cumulative shear demand"] / flexible_ns["wall length"]
flexible_ns["adjusted unit shear demand"] = flexible_ns["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ns["sheathed sides"] = 1
flexible_ns["aspect ratio"] = flexible_ns["wall height"] / flexible_ns["wall length"]

#### Check aspect ratios

We must check the aspect ratios of the shear walls to ensure we don't need to apply the aspect ratio factor in SDPWS Sec. 4.3.3.2. The maximum aspect ratio of all N-S walls is:

In [ ]:
%%render
ratio_max = check_value(flexible_ns["aspect ratio"].max(), 2, "<")

In [ ]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

In [ ]:
flexible_ns[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(flexible_ns[["adjusted unit shear demand", "sheathed sides"]], sheathing=sheathing)
)

shear_dcr_check_value = 1

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ns["shear dcr"] > shear_dcr_check_value).any():
    flexible_ns.loc[flexible_ns["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ns.loc[
        flexible_ns["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ns.loc[
            flexible_ns["shear dcr"] > shear_dcr_check_value, ["adjusted unit shear demand", "sheathed sides"]
        ],
        sheathing=sheathing,
    )

In [ ]:
# Update dataframe
flexible = flexible.reindex(columns=flexible_ns.columns)
flexible["line"] = flexible["line"].astype("string")
flexible.update(flexible_ns)

In [ ]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$w_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [lbf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [ ]:
def highlight_insufficient_capacity(row):
    styles = [""] * len(row)

    if row["$0.7 v$ [plf]"] > row["$v_{cap}$ [plf]"]:
        demand_idx = row.index.get_loc("$0.7 v$ [plf]")
        capacity_idx = row.index.get_loc("$v_{cap}$ [plf]")

        styles[demand_idx] = "background-color: red; font-weight: bold"
        styles[capacity_idx] = "background-color: red; font-weight: bold"

    return styles

In [ ]:
display_table(
    dataframe=flexible_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 4",
    label="tab:shear_wall_forces_roof_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 3",
    label="tab:shear_wall_forces_level3_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 2",
    label="tab:shear_wall_forces_level2_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 1",
    label="tab:shear_wall_forces_level1_ns",
)

#### East-West Direction

Most likely, the shear walls in the E-W direction will be governed by a rigid diaphragm assumption. 

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ew} through \autoref{tab:shear_wall_forces_level2_ew}.

The walls in the E-W direction are named to be easily found on the plan because there are only 2 lines with walls in this direction. The naming goes _[E-W line]-[N-S line intersection of wall left edge]_. For example, wall "A-C--A-3.8" is the wall on line "A-C" whose left edge begins at "A-4.2".

In [ ]:
# Plan dimensions
l_AA__AB = (30 + 9 / 12) * ft
l_AB__AC = (5 + 6 / 12) * ft
l_AC__AD = (30 + 9 / 12) * ft

d_trib_ew = L_floor

In [ ]:
# Wall tributary widths and lengths
t_w__AB = l_AA__AB + l_AB__AC / 2
t_w__AC = l_AB__AC / 2 + l_AC__AD

# t_l__AB__1 = 33.75
# t_l__AB__2 = 22.25
# t_l__AB__3 = 16
# t_l__AB__4 = 19
# t_l__AB__5 = 16
# t_l__AB__6 = 34.5
# t_l__AB__7 = 28.25
# t_l__AB__8 = 28.5
# t_l__AB__9 = 31.75
t_l__AB__1 = 33
t_l__AB__2 = 21.5
t_l__AB__3 = 16.75
t_l__AB__4 = 19.5
t_l__AB__5 = 16
t_l__AB__6 = 34
t_l__AB__7 = 28.75
t_l__AB__8 = 28
t_l__AB__9 = 32.5


# t_l__AC__1 = 44
# t_l__AC__2 = 31
# t_l__AC__3 = 27.5
# t_l__AC__4 = 31
# t_l__AC__5 = 15.5
# t_l__AC__6 = 12.34
# # t_l__AC__6 = 12.34
# # t_l__AC__7 = 19
# # t_l__AC__8 = 21.33
# t_l__AC__8 = 21.33
# t_l__AC__9 = 28.33
t_l__AC__1 = 46.67
t_l__AC__2 = 14
t_l__AC__3 = 21.5
t_l__AC__4 = 20.33
t_l__AC__5 = 30.5
t_l__AC__6 = 16
t_l__AC__7 = 29
t_l__AC__8 = 23.5
t_l__AC__9 = 28.5

In [ ]:
# Wall segment lengths
# l_AB__1 = 21
# l_AB__2 = 13.25
# l_AB__3 = 9
# l_AB__4 = 12
# l_AB__5 = 11
# l_AB__6 = 29.75
# l_AB__7 = 12.75
# l_AB__8 = 16.25
# l_AB__9 = 13.75
l_AB__1 = 20
l_AB__2 = 13
l_AB__3 = 9
l_AB__4 = 12
l_AB__5 = 10.67
l_AB__6 = 29
l_AB__7 = 13
l_AB__8 = 11.67
l_AB__9 = 10

# l_AC__1 = 17.25
# l_AC__2 = 23
# l_AC__3 = 22.5
# l_AC__4 = 26
# # l_AC__5 = 11.25
# l_AC__5 = 16
# l_AC__6 = 7
# l_AC__7 = 6.75
# l_AC__8 = 10.25
# l_AC__9 = 25.25
l_AC__1 = 16
l_AC__2 = 9.34
l_AC__3 = 16.75
l_AC__4 = 15
l_AC__5 = 26
l_AC__6 = 11.5
l_AC__7 = 16
l_AC__8 = 9.75
l_AC__9 = 24

In [ ]:
flexible_ew.loc["Level 4", "tributary weight"] = float(f_p__roof)
flexible_ew.loc["Level 3", "tributary weight"] = float(f_p__4)
flexible_ew.loc["Level 2", "tributary weight"] = float(f_p__3)
flexible_ew.loc["Level 1", "tributary weight"] = float(f_p__2)

# Shear wall height is for the floor below
flexible_ew.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ew.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ew.loc["Level 2", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ew.loc["Level 1", "wall height"] = float(seismic_loads.loc["Level 1", "floor height"])

# Add wall tributary widths and wall lengths to df
wall_tributary = {
    "A-B--A-1": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__1, "wall_length": l_AB__1},
    "A-B--A-2.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__2, "wall_length": l_AB__2},
    "A-B--A-3.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__3, "wall_length": l_AB__3},
    "A-B--A-4": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__4, "wall_length": l_AB__4},
    "A-B--A-4.8": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__5, "wall_length": l_AB__5},
    "A-B--A-5.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__6, "wall_length": l_AB__6},
    "A-B--A-6.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__7, "wall_length": l_AB__7},
    "A-B--A-7.9": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__8, "wall_length": l_AB__8},
    "A-B--A-8.8": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__9, "wall_length": l_AB__9},
    "A-C--A-1.8": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__1, "wall_length": l_AC__1},
    "A-C--A-3.2": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__2, "wall_length": l_AC__2},
    "A-C--A-3.8": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__3, "wall_length": l_AC__3},
    "A-C--A-4.6": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__7, "wall_length": l_AC__7},
    "A-C--A-5.5": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__4, "wall_length": l_AC__4},
    "A-C--A-6.5": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__5, "wall_length": l_AC__5},
    "A-C--A-6.9": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__6, "wall_length": l_AC__6},
    "A-C--A-8.5": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__8, "wall_length": l_AC__8},
    "A-C--A-9.2": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__9, "wall_length": l_AC__9},
}

flexible_ew["wall length"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary width"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary depth"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary area"] = flexible_ew["tributary depth"] * flexible_ew["tributary width"]

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ew["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up.\n\
    Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ew["line"] = flexible_ew.index.get_level_values("Wall").str.split("--").str[0]

# Get tributary area and wall length for the shear line
flexible_ew["line tributary area"] = flexible_ew.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ew["line wall length"] = flexible_ew.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ew["line shear demand"] = flexible_ew["line tributary area"] * flexible_ew["tributary weight"]
flexible_ew["line unit shear"] = flexible_ew["line shear demand"] / flexible_ew["line wall length"]
flexible_ew["line cumulative shear demand"] = flexible_ew.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ew["line cumulative unit shear"] = (
    flexible_ew["line cumulative shear demand"] / flexible_ew["line wall length"]
)

flexible_ew["shear demand"] = flexible_ew["line unit shear"] * flexible_ew["wall length"]
flexible_ew["cumulative shear demand"] = flexible_ew.groupby(level="Wall")["shear demand"].cumsum()
flexible_ew["unit shear demand"] = flexible_ew["cumulative shear demand"] / flexible_ew["wall length"]
flexible_ew["adjusted unit shear demand"] = flexible_ew["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ew["sheathed sides"] = 1
flexible_ew["aspect ratio"] = flexible_ew["wall height"] / flexible_ew["wall length"]

#### Check aspect ratios

We will also check the aspect ratios of the E-W walls.

In [ ]:
%%render
ratio_max = check_value(flexible_ew["aspect ratio"].max(), 2, "<")

In [ ]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

In [ ]:
# using sheathing from the NS calc in memory

flexible_ew[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        flexible_ew[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ew["shear dcr"] > shear_dcr_check_value).any():
    flexible_ew.loc[flexible_ew["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ew.loc[
        flexible_ew["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ew.loc[
            flexible_ew["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

In [ ]:
# Update dataframe
flexible.update(flexible_ew)

In [ ]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$W_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [ ]:
display_table(
    dataframe=flexible_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 4",
    label="tab:shear_wall_forces_roof_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 3",
    label="tab:shear_wall_forces_level3_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 2",
    label="tab:shear_wall_forces_level2_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 1",
    label="tab:shear_wall_forces_level1_ew",
)

### Rigid Diaphragm Assumption

With a rigid diaphragm assumption, we now must calculate the following in each direction:

1. The relative stiffnesses of the shear walls to obtain a center of rigidity.
2. The center of mass of the diaphragm with the center of rigidity to obtain an eccentricity.

The relative stiffnesses of the shear walls can be calculated by using the deflection equation from NDS SDPWS Sec. 4.3.4

\begin{align*}
k &= \frac{F}{\delta},\, \text{assume }F = \text{induced load demand in the shear wall} \\
\delta_{sw} &= \frac{8vh^3}{EAb} + \frac{vh}{1000 G_a} + \frac{h \Delta_A}{b}\quad \text{(SDPWS Eq. 4.3-1)}
\end{align*}

where

- $b =$ the shear wall length [ft]
- $\Delta_A =$ vertical deformation of the wall overturning anchorage system + vertical compression deformation [in]
- $E =$ modulus of elasticity of end posts [psi]
- $A =$ area of end post cross-section [in$^2$]
- $G_a =$ apparent shear wall shear stiffness from nail slip and panel shear deformation [kip/in]
- $h =$ shear wall height [ft]
- $v =$ unit shear force induced by the design load [plf]
- $\delta_{sw} =$ maximum shear wall deflection determined by elastic analysis [in]

Note that for the following calculations, we will conservatively use the unit shear force induced by the nominal load, not the design load, for the value of $v$. This is because the stiffnesses obtained from this use will allow us to use the stiffnesses for deflection limits and story drift calculations

We will assume that the end posts are (2) 2x4 Douglas Fir No. 2; this means:

In [ ]:
%%render
E_endpost = 1_600_000 * psi
A_endpost = 2 * 5.25 * inch**2

We will assume a continuous tiedown rod system. A conservative first-pass value for $\Delta_A$ can be:

In [ ]:
%%render params
Delta_A = 0.25 * inch

#### Shear Wall Rigidities

With the deflections of each shear wall, along with the force demand of each shear wall, we can obtain the stiffnesses (and therefore, relative stiffnesses) of all the walls.

Each relative stiffness is relative to the level. The sum of all relative stiffnesses on each level is $1.0$.

\begin{align*}
k &= \frac{F_x}{\delta_{sw}} \\
R &= \frac{k_{wall}}{\sum {k_{wall}}}
\end{align*}

##### North-South Direction

Rigidities in the N-S direction can be found in \autoref{tab:wall_rigidities_ns_roof} through \autoref{tab:wall_rigidities_ns_level2}.

In [ ]:
rigid_ns["unit shear demand"] = flexible_ns["unit shear demand"]
rigid_ns["wall length"] = flexible_ns["wall length"]
rigid_ns["wall height"] = flexible_ns["wall height"]
rigid_ns["shear stiffness"] = flexible_ns["shear stiffness"]
rigid_ns["sheathed sides"] = flexible_ns["sheathed sides"]
rigid_ns["nail spacing"] = flexible_ns["nail spacing"]
rigid_ns["end post youngs modulus"] = float(E_endpost)
rigid_ns["end post area"] = float(A_endpost)
rigid_ns["Delta_A"] = float(Delta_A)

rigid_ns["delta_sw"] = (
    8
    * rigid_ns["unit shear demand"]
    * rigid_ns["wall height"] ** 3
    / (rigid_ns["end post youngs modulus"] * rigid_ns["end post area"] * rigid_ns["wall length"])
    + rigid_ns["unit shear demand"]
    * rigid_ns["wall height"]
    / (1000 * rigid_ns["shear stiffness"] * rigid_ns["sheathed sides"])
    + rigid_ns["wall height"] * rigid_ns["Delta_A"] / rigid_ns["wall length"]
)

In [ ]:
rigid_ns["cumulative shear demand"] = flexible_ns["cumulative shear demand"]
rigid_ns["wall stiffness"] = rigid_ns["cumulative shear demand"] / rigid_ns["delta_sw"]
rigid_ns["total level stiffness"] = rigid_ns.groupby("Level")["wall stiffness"].transform("sum")
rigid_ns["total level cumulative shear"] = rigid_ns.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ns["relative wall stiffness"] = rigid_ns["wall stiffness"] / rigid_ns["total level stiffness"]
rigid_ns["direct shear force"] = rigid_ns["total level cumulative shear"] * rigid_ns["relative wall stiffness"]

In [ ]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [ ]:
display_table(
    dataframe=rigid_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 4",
    label="tab:wall_rigidities_ns_level4",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 3",
    label="tab:wall_rigidities_ns_level3",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 2",
    label="tab:wall_rigidities_ns_level2",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 1",
    label="tab:wall_rigidities_ns_level1",
)

##### East-West Direction

Rigidities in the E-W direction can be found in \autoref{tab:wall_rigidities_ew_roof}, \autoref{tab:wall_rigidities_ew_level4}, \autoref{tab:wall_rigidities_ew_level3}, \autoref{tab:wall_rigidities_ew_level2},

In [ ]:
rigid_ew["unit shear demand"] = flexible_ew["unit shear demand"]
rigid_ew["wall length"] = flexible_ew["wall length"]
rigid_ew["wall height"] = flexible_ew["wall height"]
rigid_ew["shear stiffness"] = flexible_ew["shear stiffness"]
rigid_ew["nail spacing"] = flexible_ew["nail spacing"]
rigid_ew["sheathed sides"] = flexible_ew["sheathed sides"]
rigid_ew["end post youngs modulus"] = float(E_endpost)
rigid_ew["end post area"] = float(A_endpost)
rigid_ew["Delta_A"] = float(Delta_A)

rigid_ew["delta_sw"] = (
    8
    * rigid_ew["unit shear demand"]
    * rigid_ew["wall height"] ** 3
    / (rigid_ew["end post youngs modulus"] * rigid_ew["end post area"] * rigid_ew["wall length"])
    + rigid_ew["unit shear demand"]
    * rigid_ew["wall height"]
    / (1000 * rigid_ew["shear stiffness"] * rigid_ew["sheathed sides"])
    + rigid_ew["wall height"] * rigid_ew["Delta_A"] / rigid_ew["wall length"]
)

In [ ]:
rigid_ew["cumulative shear demand"] = flexible_ew["cumulative shear demand"]
rigid_ew["wall stiffness"] = rigid_ew["cumulative shear demand"] / rigid_ew["delta_sw"]
rigid_ew["total level stiffness"] = rigid_ew.groupby("Level")["wall stiffness"].transform("sum")
rigid_ew["total level cumulative shear"] = rigid_ew.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ew["relative wall stiffness"] = rigid_ew["wall stiffness"] / rigid_ew["total level stiffness"]
rigid_ew["direct shear force"] = rigid_ew["total level cumulative shear"] * rigid_ew["relative wall stiffness"]

In [ ]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [ ]:
display_table(
    dataframe=rigid_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 4",
    label="tab:wall_rigidities_ew_level4",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 3",
    label="tab:wall_rigidities_ew_level3",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 2",
    label="tab:wall_rigidities_ew_level2",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 1",
    label="tab:wall_rigidities_ew_level1",
)

#### Center of Rigidity and Center of Mass

We can find the center of rigidity by taking the weighted sum of all wall stiffnesses and their locations relative to a datum. In this case, we will use the north-west-most corner of the building (i.e., the intersection of lines A-1 and A-A).

$$
\bar{x}_r = \frac{\sum k_{ns} x}{\sum k_{ns}},\, \bar{y}_r = \frac{\sum k_{ew} y}{\sum k_{ew}}
$$

Note: each wall is assumed to have zero rigidity in the direction perpendicular to them.

The building is symmetric in each direction when projected to that direction, so the centers of mass are just half of each total plan dimension.

In [ ]:
%%render
X_plan = L_floor + L_jog
Y_plan = w_building
CM_x = X_plan / 2 - 0.004
CM_y = Y_plan / 2 - 0.004

An accidental torsion eccentricity of 5% must be added/subtracted.

In [ ]:
%%render
CM_x__a__plus = CM_x + X_plan * 0.05
CM_x__a__minus = CM_x - X_plan * 0.05
CM_y__a__plus = CM_y + Y_plan * 0.05
CM_y__a__minus = CM_y - Y_plan * 0.05

The eccentricity used is determined by the location of the wall in relation to the center of rigidity. Since the wall rotates about the center of rigidity, we want to position the center of mass to ensure each wall is getting the maximum possible force to be conservative. If the center of mass is positioned on the opposite side of the center of rigidity relative to the wall in question, that mass will _decrease_ the shear experienced by that wall.

This means we always want the center of mass to be on the same side of the center of rigidity as the wall. The eccentricities shown in the following tables are based on the worst-case scenario. This is not immediately obvious due to all torques being given a positive value. 

How the center of rigidity, center of mass, and eccentricity interact are illustrated in \autoref{fig:rigid_diaphragm_moments}. The moments generated by the rotation of the center of mass around the center of rigidity differ based on the chosen eccentricity.

![Moments applied by a rigid diaphragm](images/rigid_diaphragm_moments.png){width=50% #fig:rigid_diaphragm_moments}

$$
d_x = \left| x - \text{CR}_x \right|\text{ and }\, d_y = \left| y - \text{CR}_y \right|
$$

In [ ]:
# Update dataframe
rigid = rigid.reindex(columns=rigid_ns.columns)
rigid.update(rigid_ns)
rigid.update(rigid_ew)

In [ ]:
# Location of wall lines of rigidity
wall_rigidity_loc = {
    # N-S
    "A-2": {"x": float(l_A1__A2), "y": 0},
    "A-2.4": {"x": float(l_A1__A24), "y": 0},
    "A-3": {"x": float(l_A1__A2 + l_A2__A3), "y": 0},
    "A-3.7": {"x": float(l_A1__A2 + l_A2__A3 + l_A3__A37), "y": 0},
    "A-4": {"x": float(l_A1__A2 + l_A2__A3 + l_A3__A4), "y": 0},
    "A-5": {"x": float(l_A1__A2 + l_A2__A3 + l_A3__A4 + l_A4__A5), "y": 0},
    "A-5.1": {"x": float(l_A1__A24 + l_A24__A35 + l_A35__A51), "y": 0},
    "A-6": {"x": float(l_A1__A2 + l_A2__A3 + l_A3__A4 + l_A4__A5 + l_A5__A6), "y": 0},
    "A-6.1": {"x": float(l_A1__A24 + l_A24__A35 + l_A35__A51 + l_A51__A61), "y": 0},
    "A-6.8": {"x": float(l_A1__A24 + l_A24__A35 + l_A35__A51 + l_A51__A61 + l_A61__A68), "y": 0},
    "A-7": {"x": float(l_A1__A2 + l_A2__A3 + l_A3__A4 + l_A4__A5 + l_A5__A6 + l_A6__A7), "y": 0},
    "A-7.8": {"x": float(l_A1__A24 + l_A24__A35 + l_A35__A51 + l_A51__A61 + l_A61__A68 + l_A68__A78), "y": 0},
    "A-8": {"x": float(l_A1__A2 + l_A2__A3 + l_A3__A4 + l_A4__A5 + l_A5__A6 + l_A6__A7 + l_A7__A8), "y": 0},
    "A-8.2": {
        "x": float(l_A1__A24 + l_A24__A35 + l_A35__A51 + l_A51__A61 + l_A61__A68 + l_A68__A78 + l_A78__A82),
        "y": 0,
    },
    "A-8.9": {
        "x": float(
            l_A1__A24 + l_A24__A35 + l_A35__A51 + l_A51__A61 + l_A61__A68 + l_A68__A78 + l_A78__A82 + l_A82__A89
        ),
        "y": 0,
    },
    "A-9": {"x": float(l_A1__A2 + l_A2__A3 + l_A3__A4 + l_A4__A5 + l_A5__A6 + l_A6__A7 + l_A7__A8 + l_A8__A9), "y": 0},
    # E-W
    "A-B--A-1": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-2.5": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-3.5": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-4": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-4.8": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-5.5": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-6.5": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-7.9": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-8.8": {"x": 0, "y": float(l_AA__AB)},
    "A-C--A-1.8": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-3.2": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-3.8": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-4.6": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-5.5": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-6.5": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-6.9": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-8.5": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-9.2": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
}

In [ ]:
# Calc centers of rigidity
rigid["x"] = rigid.index.get_level_values("Wall").map({key: value["x"] for key, value in wall_rigidity_loc.items()})
rigid["y"] = rigid.index.get_level_values("Wall").map({key: value["y"] for key, value in wall_rigidity_loc.items()})

rigid["kx"] = rigid["relative wall stiffness"] * rigid["x"]
rigid["CRx"] = rigid.groupby(["Level", "Direction"])["kx"].transform("sum")  # all relative wall rigidities sum to 1
rigid["ky"] = rigid["relative wall stiffness"] * rigid["y"]
rigid["CRy"] = rigid.groupby(["Level", "Direction"])["ky"].transform("sum")  # all relative wall rigidities sum to 1

# x eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
plan_dim_x = float(X_plan)
CMx = plan_dim_x / 2
CMxa_plus = CMx + plan_dim_x * 0.05
CMxa_minus = CMx - plan_dim_x * 0.05
rigid["ex"] = np.where(rigid["x"] <= rigid["CRx"], rigid["CRx"] - CMxa_minus, CMxa_plus - rigid["CRx"])
rigid["ex"] = np.where(rigid["ex"] <= 0, 0.05 * plan_dim_x, rigid["ex"])  # assign 0s to negatives

# y eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
plan_dim_y = float(Y_plan)
CMy = plan_dim_y / 2
CMya_plus = CMy + plan_dim_y * 0.05
CMya_minus = CMy - plan_dim_y * 0.05
rigid["ey"] = np.where(rigid["y"] <= rigid["CRy"], rigid["CRy"] - CMya_minus, CMya_plus - rigid["CRy"])
rigid["ey"] = np.where(rigid["ey"] <= 0, 0.05 * plan_dim_y, rigid["ey"])  # assign 0s to negatives

# distance from CR
rigid["dx"] = abs(rigid["x"] - rigid["CRx"])
rigid["dy"] = abs(rigid["y"] - rigid["CRy"])

# Torque
rigid["torque x"] = rigid["total level cumulative shear"] * rigid["ey"]
rigid["torque y"] = rigid["total level cumulative shear"] * rigid["ex"]

# Polar moment of inertia
rigid["Jx wall"] = rigid["relative wall stiffness"] * rigid["dx"] ** 2
rigid["Jy wall"] = rigid["relative wall stiffness"] * rigid["dy"] ** 2
rigid["J"] = (rigid["Jx wall"] + rigid["Jy wall"]).groupby(["Level"]).transform("sum")

rigid["torsional shear force x"] = rigid["torque x"] * rigid["relative wall stiffness"] * rigid["dx"] / rigid["J"]
rigid["torsional shear force y"] = rigid["torque y"] * rigid["relative wall stiffness"] * rigid["dy"] / rigid["J"]
rigid["torsional shear force"] = np.where(
    rigid["torsional shear force x"] == 0, rigid["torsional shear force y"], rigid["torsional shear force x"]
)
rigid["shear force demand"] = rigid["torsional shear force"] + rigid["direct shear force"]

The centers of rigidity for each floor are given in \autoref{tab:center_of_rigidity_roof} through \autoref{tab:center_of_rigidity_level2}.

In [ ]:
column_name_map = {
    "x": "$x$ [ft]",
    "y": "$y$ [ft]",
    "CRx": "CR$_x$ [ft]",
    "dx": "$d_x$ [ft]",
    "ex": "e$_x$ [ft]",
    "CRy": "CR$_y$ [ft]",
    "dy": "$d_y$ [ft]",
    "ey": "e$_y$ [ft]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 4",
    label="tab:center_of_rigidity_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 3",
    label="tab:center_of_rigidity_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 2",
    label="tab:center_of_rigidity_level2",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 1",
    label="tab:center_of_rigidity_level1",
)

And now, we can get the forces in the shear walls using the rigid diaphragm assumption in \autoref{tab:rigid_forces_roof} through \autoref{tab:rigid_forces_level2}. These forces can be compared to the flexible shear wall forces obtained earlier, $F_{flex}$.

In [ ]:
rigid["flexible shear force demand"] = flexible["cumulative shear demand"]

In [ ]:
column_name_map = {
    "torque x": "$T_x$ [lbf-ft]",
    "torque y": "$T_y$ [lbf-ft]",
    "dx": "$d_x$ [ft]",
    "dy": "$d_y$ [ft]",
    "relative wall stiffness": "$R$ [-]",
    "torsional shear force": "$F_t$ [lbf]",
    "direct shear force": "$F_v$ [lbf]",
    "shear force demand": "$F_{tot}$ [lbf]",
    "flexible shear force demand": "$F_{flex}$ [lbf]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 4",
    label="tab:rigid_forces_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 3",
    label="tab:rigid_forces_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 2",
    label="tab:rigid_forces_level2",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 1",
    label="tab:rigid_forces_level1",
)

### Final Shear Wall Nailing Selection

With the envelope procedure complete, we can now select the shear wall sheathing and nailing specifications in \autoref{tab:shear_walls_roof} through \autoref{tab:shear_walls_level2}.

In [ ]:
shear_walls["wall length"] = rigid["wall length"]
shear_walls["rigid shear force demand"] = rigid["shear force demand"]
shear_walls["flexible shear force demand"] = rigid["flexible shear force demand"]
shear_walls["shear force demand"] = np.where(
    shear_walls["flexible shear force demand"] >= shear_walls["rigid shear force demand"],
    shear_walls["flexible shear force demand"],
    shear_walls["rigid shear force demand"],
)
shear_walls["unit shear demand"] = shear_walls["shear force demand"] / shear_walls["wall length"]
shear_walls["adjusted unit shear demand"] = (
    shear_walls["shear force demand"] / shear_walls["wall length"] * LOAD_FACTOR_SEISMIC_ASD
)
shear_walls["sheathed sides"] = 1

# using sheathing from the NS calc in memory

shear_walls[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        shear_walls[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (shear_walls["shear dcr"] > shear_dcr_check_value).any():
    shear_walls.loc[shear_walls["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    shear_walls.loc[
        shear_walls["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        shear_walls.loc[
            shear_walls["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

column_name_map = {
    "shear force demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    caption="Final shear wall design values - Level 4",
    label="tab:shear_walls_level4",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    caption="Final shear wall design values - Level 3",
    label="tab:shear_walls_level3",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    caption="Final shear wall design values - Level 2",
    label="tab:shear_walls_level2",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position_float="centering",
    caption="Final shear wall design values - Level 1",
    label="tab:shear_walls_level1",
)

### Shear Wall Axial Forces

#### Tension Forces

The shear forces get converted to overturning forces in the end posts of the shear walls. This overturning moment is counteracted by the dead load tributary to each shear wall according to ASCE 7-16 Sec. 2.4.5. The applicable equation is:

$$
0.6 D - 0.7 E_v + 0.7 E_h \quad\text{(ASCE 7-16 Eq. 2.4.5-10)}
$$

There are no vertical seismic effects, $E_v$, so we will only consider the horizontal seismic effects, $E_h$, and dead loads, $D$.

The overturning moment is:

$$
M_{OT} = V h_{wall}
$$

Using the area dead loads from earlier, along with the tributary areas for each shear wall, we can get the dead load that is applied to each wall. This resists the overturning moment at the centroid of the wall.

$$
M_R = \frac{D l_{wall}}{2}
$$

We will assume that the holddowns are 6" inset from the end posts. The tension caused by horizontal load effects counteracted by dead loads is:

$$
T_{seismic} = \frac{M_{OT} - M_R}{l_{wall} - 0.5}\quad\text{(units in feet)}
$$

In [ ]:
shear_walls["line"] = flexible["line"]
shear_walls["wall height"] = rigid["wall height"]
shear_walls["tributary area"] = flexible["tributary area"]
shear_walls["floor OTM"] = shear_walls["shear force demand"] * shear_walls["wall height"]
shear_walls["OTM"] = shear_walls.groupby(level="Wall")["floor OTM"].cumsum()
shear_walls["dead load"] = 0.6 * float(U_floor * 1000) * shear_walls["tributary area"]

# Reduce resisting moment dead load in some walls that definitely are not correct
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "A-2", "dead load"] *= 1 / 2
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "A-2.4", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "A-3.5", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "A-5.1", "dead load"] *= 1 / 2
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "A-6", "dead load"] *= 1 / 2
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "A-6.8", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "A-7", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "A-7.8", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "A-8", "dead load"] *= 1 / 2
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "A-8.9", "dead load"] *= 1 / 2
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "A-9", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls["line"] == "A-B", "dead load"] *= 0
shear_walls.loc[shear_walls["line"] == "A-C", "dead load"] *= 0

shear_walls["RM"] = shear_walls["dead load"] * shear_walls["wall length"] / 2

shear_walls["total moment"] = shear_walls["OTM"] - shear_walls["RM"]
shear_walls["total moment"] = np.where(
    shear_walls["RM"] <= shear_walls["OTM"], shear_walls["OTM"] - shear_walls["RM"], 0
)
wall_arm_shortening_length = 0.5  # feet due to holddown positioning
shear_walls["tension force"] = shear_walls["total moment"] / (shear_walls["wall length"] - wall_arm_shortening_length)
num_bins = 5
shear_walls["binned tension force"] = (
    shear_walls["tension force"]
    .groupby(level="Level")
    .transform(lambda s: pd.qcut(s, q=num_bins, duplicates="drop").map(lambda x: x.right if pd.notna(x) else 0))
)

#### Compression Forces

We will calculate the compression forces in a very similar way. However, we will ignore the resisting moment.

The live load, $L$, is $40$ psf, while the snow load, $S$, is $25$ psf and applies only to corridor walls as that is how the roof trusses are supported.

$$
C_{seismic} = \frac{M_{OT}}{l_{wall} - 0.5}\quad\text{(units in feet)}
$$

In [ ]:
shear_walls["compression force"] = shear_walls["OTM"] / (shear_walls["wall length"] - wall_arm_shortening_length)
shear_walls["binned compression force"] = (
    shear_walls["compression force"]
    .groupby(level="Level")
    .transform(lambda s: pd.qcut(s, q=num_bins).map(lambda x: x.right if pd.notna(x) else 0))
    # .transform(lambda s: pd.qcut(s, q=num_bins, duplicates="drop").map(lambda x: x.right if pd.notna(x) else pd.NA))
)

#### Holddown Forces Table

With the tension and compression forces calculated, we can construct a table of the forces for each level. We will default to 5 bins for each level to give us options A through E.

However, there is only one table in the plans for all the holddown forces, so these will all get compared together as an aggregate of all buildings and simplified further.

In [ ]:
NUM_OPTIONS = 5
LABELS = list("ABCDE")

schedule_rows = []


def assign_schedule(group):
    group = group.copy()

    level = group.index.get_level_values("Level")[0]

    n = len(group)
    group["_bin"] = np.floor(np.arange(n) * min(NUM_OPTIONS, n) / n)

    schedule = (
        group.groupby("_bin")
        .agg(
            tension=("tension force", "max"),
            compression=("compression force", "max"),
        )
        .reset_index()
    )

    # convert to kips
    schedule["tension"] /= 1000
    schedule["compression"] /= 1000

    # sort ONLY schedule
    schedule["governing"] = schedule[["tension", "compression"]].max(axis=1)
    schedule = schedule.sort_values("governing").reset_index(drop=True)

    # assign A–E
    schedule["option"] = LABELS[: len(schedule)]

    # IMPORTANT: keep level in schedule
    schedule["Level"] = level

    # map bins → options
    bin_to_option = dict(zip(schedule["_bin"], schedule["option"]))
    group["option"] = group["_bin"].map(bin_to_option)

    schedule_map = schedule.set_index("option")[["tension", "compression"]]

    group["binned tension force"] = group["option"].map(schedule_map["tension"])
    group["binned compression force"] = group["option"].map(schedule_map["compression"])

    schedule_rows.append(schedule[["Level", "option", "tension", "compression"]])

    return group.drop(columns=["_bin"])


shear_walls = shear_walls.groupby(level="Level", group_keys=False).apply(assign_schedule)

schedule_df = (
    pd.concat(schedule_rows, ignore_index=True)
    .rename(
        columns={
            "tension": "binned tension force",
            "compression": "binned compression force",
        }
    )
    .set_index(["Level", "option"])
    .sort_index()
)

In [ ]:
column_name_map = {
    "binned tension force": "$T$ [kip]",
    "binned compression force": "$C$ [kip]",
}
display_table(
    schedule_df,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Holddown Schedule",
    label="tab:holddown_schedule",
)

## Diaphragm Design

We must first get the diaphragm inertial design forces. These are distinct from the seismic forces determined in \autoref{tab:seismic_forces_calculation_table}.

\begin{align*}
F_{px} &= \frac{\sum_{i=x}^n F_i}{\sum_{i=x}^n w_i} w_{px} \quad \text{(ASCE Eq. 12.10-1)} \\
F_{px} &\ge 0.2 S_{DS} I_e w_{px} \quad \text{(ASCE Eq. 12.10-2)} \\
F_{px} &\le 0.4 S_{DS} I_e w_{px}  \quad \text{(ASCE Eq. 12.10-3)}
\end{align*}

These forces are shown in \autoref{tab:diaphragm_forces}

In [ ]:
seismic_loads["cumulative floor weight"] = seismic_loads.iloc[::-1]["floor weight"].cumsum()
seismic_loads["F_x,cum"] = seismic_loads.iloc[::-1]["F_x"].cumsum()
seismic_loads["F_px lower bound"] = 0.2 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px upper bound"] = 0.4 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px initial"] = (
    seismic_loads["F_x,cum"] / seismic_loads["cumulative floor weight"] * seismic_loads["floor weight"]
)

# Assign F_px with limits from ASCE
seismic_loads["F_px"] = np.where(
    seismic_loads["F_px initial"] <= seismic_loads["F_px lower bound"],
    seismic_loads["F_px lower bound"],
    np.where(
        seismic_loads["F_px initial"] >= seismic_loads["F_px upper bound"],
        seismic_loads["F_px upper bound"],
        seismic_loads["F_px initial"],
    ),
)

In [ ]:
column_name_map = {
    "floor weight": "$w_{px}$ [kip]",
    "cumulative floor weight": r"$\sum w_x$ [kip]",
    "F_x,cum": r"$\sum F_x$ [kip]",
    "F_px initial": "$F_{px,init}$ [kip]",
    "F_px lower bound": "$F_{px,lower}$ [kip]",
    "F_px upper bound": "$F_{px,upper}$ [kip]",
    "F_px": "$F_{px}$ [kip]",
}
display_table(
    dataframe=seismic_loads.iloc[:0:-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Forces",
    label="tab:diaphragm_forces",
)

With these diaphragm forces, we can design the diaphragm for the worst-case cantilever and simply-supported conditions.

The longest simply-supported span in the N-S direction is about 34.5 feet, while the longest cantilever span is about 32 feet. The depth of the diaphragm in the N-S direction is 67 feet. The longest simply-supported span in the E-W direction is about 5.5 feet, while the longest cantilever span is about 31 feet. The depth of the diaphragm in the E-W direction is 230 feet.

Diaphragm unit shear demands are shown in \autoref{tab:diaphragm_unit_shears}

In [ ]:
# Set up diaphragm dataframe
levels = ["Level 4", "Level 3", "Level 2", "Level 1"]
directions = ["N-S", "E-W"]
index = pd.MultiIndex.from_product(
    [levels, directions],
    names=["Level", "Direction"],
)
diaphragms = pd.DataFrame(index=index)
diaphragms_ns = diaphragms.xs("N-S", level="Direction").copy()
diaphragms_ew = diaphragms.xs("E-W", level="Direction").copy()

In [ ]:
diaphragms_ns["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ns["area load"] = diaphragms_ns["F_px"] / float(A_level)
diaphragms_ns["L_simp"] = 34.5
diaphragms_ns["L_cant"] = 32
diaphragms_ns["depth"] = 67
diaphragms_ns["V_simp"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_simp"] / 2
diaphragms_ns["V_cant"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_cant"]
diaphragms_ns["v_simp"] = diaphragms_ns["V_simp"] / diaphragms_ns["depth"]
diaphragms_ns["v_cant"] = diaphragms_ns["V_cant"] / diaphragms_ns["depth"]

We will assume the diaphragm sheathing has the following properties:

- Unblocked
- WSP Sheathing
- 15/32" thick
- 10d common nails
- Plywood
- Nailed face is 2"
- Edge adjoining cases 2 through 6

The diaphragm capacity checks for this chosen sheathing are shown in \autoref{tab:diaphragm_capacities}

In [ ]:
# Set baseline sheathing properties. We set the number of sheathed sides later
from structural_tools.seismic.sheathing import SheathingApplication

diaphragm_sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
    minimum_nominal_width_of_nailed_face=2,
    adjoining_panel_edge_location=(2, 3, 4, 5, 6),
)

diaphragms_ns[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ns["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ns[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ns["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [ ]:
diaphragms_ew["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ew["area load"] = diaphragms_ew["F_px"] / float(A_level)
diaphragms_ew["L_simp"] = 5.5
diaphragms_ew["L_cant"] = 31
diaphragms_ew["depth"] = float(L_floor)
diaphragms_ew["V_simp"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_simp"] / 2
diaphragms_ew["V_cant"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_cant"]
diaphragms_ew["v_simp"] = diaphragms_ew["V_simp"] / diaphragms_ew["depth"]
diaphragms_ew["v_cant"] = diaphragms_ew["V_cant"] / diaphragms_ew["depth"]

In [ ]:
diaphragms_ew[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ew["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ew[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ew["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [ ]:
idx = pd.IndexSlice
diaphragms = diaphragms.reindex(columns=diaphragms_ns.columns)
diaphragms_ns["Direction"] = "N-S"
diaphragms_ew["Direction"] = "E-W"
diaphragms_ns = diaphragms_ns.set_index("Direction", append=True)
diaphragms_ew = diaphragms_ew.set_index("Direction", append=True)
diaphragms.update(diaphragms_ns)
diaphragms.update(diaphragms_ew)

In [ ]:
column_name_map = {
    "F_px": "$F_{px}$ [lbf]",
    "area load": "$w$ [psf]",
    "V_simp": "$V_{simp}$ [lbf]",
    "V_cant": "$V_{cant}$ [lbf]",
    "v_simp": "$v_{simp}$ [plf]",
    "v_cant": "$v_{cant}$ [plf]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Unit Shear Demands",
    label="tab:diaphragm_unit_shears",
)

In [ ]:
column_name_map = {
    "v_simp": "$v_{simp}$ [plf]",
    "adjusted unit shear capacity simply supported": "$v_{cap,simp}$ [plf]",
    "shear dcr simply supported": "DCR$_{simp}$ [-]",
    "v_cant": "$v_{cant}$ [plf]",
    "adjusted unit shear capacity cantilever": "$v_{cap,cant}$ [plf]",
    "shear dcr cantilever": "DCR$_{cant}$ [-]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Capacity Checks",
    label="tab:diaphragm_capacities",
)

In [ ]:
if (
    (diaphragms[["shear dcr simply supported", "shear dcr cantilever"]] <= 1).all().all()
):  # double .all() because 2 columns
    display_text(r"$\therefore$ All shear capacities are OK")
else:
    display_text("You must check for new diaphragm nailing.")

## Story Drift Determination

The inelastic design story drift is calculated as the difference in deflections between stories at their centers of mass.

We can find the shear wall line that is closest to the center of mass, then take the cumulative force on that line and divide it by the cumulative wall stiffness on that line. This will give us an approximate elastic story drift that will then be converted to inelastic story drift with $C_d$ and $I_e$.

Note that this is a conservative estimation of inelastic story drift due to assuming the deflection of a single shear wall line, rather than the full deflection of the diaphragm as a rigid unit.

\begin{align*}
    \delta_{xe} &= F/k \\
    \delta_x &= \frac{C_d \delta_{xe}}{I_e}
\end{align*}

In [ ]:
%%render
C_d
I_e

In [ ]:
shear_walls["line"] = flexible["line"]
shear_walls["wall stiffness"] = rigid["wall stiffness"]
shear_walls["cumulative shear demand"] = flexible["cumulative shear demand"]
shear_walls["x"] = rigid["x"]
shear_walls["y"] = rigid["y"]
shear_walls["delta_sw"] = rigid["delta_sw"]


def line_deflection(group, line):
    line_walls = group[group["line"] == line]

    total_stiffness = line_walls["wall stiffness"].sum()
    total_shear = line_walls["cumulative shear demand"].sum()

    return total_shear / total_stiffness


def get_cm_line_deflection(group, CM_x, CM_y, C_d, I_e):
    x_wall = group.loc[(group["x"] - CM_x).abs().idxmin()]

    y_wall = group.loc[(group["y"] - CM_y).abs().idxmin()]

    x_line = x_wall["line"]
    y_line = y_wall["line"]

    x_delta_sw = line_deflection(group, x_line)
    y_delta_sw = line_deflection(group, y_line)

    drift_factor = C_d / I_e

    return pd.Series(
        {
            "x line": x_line,
            "x delta_sw": x_delta_sw,
            "x inelastic drift": x_delta_sw * drift_factor,
            "y line": y_line,
            "y delta_sw": y_delta_sw,
            "y inelastic drift": y_delta_sw * drift_factor,
        }
    )


story_drift = shear_walls.groupby(level="Level", sort=False).apply(
    get_cm_line_deflection,
    CM_x=CM_x,
    CM_y=CM_y,
    C_d=C_d,
    I_e=I_e,
)
story_drift["allowable story drift"] = seismic_loads["floor height"] * 0.02 * 12

In [ ]:
column_name_map = {
    "x line": "N-S line",
    "x delta_sw": r"$\delta_{xe,ns}$ [in]",
    "x inelastic drift": r"$\delta_{x,ns}$ [in]",
    "y line": "E-W line",
    "y delta_sw": r"$\delta_{xe,ew}$ [in]",
    "y inelastic drift": r"$\delta_{x,ew}$ [in]",
    "allowable story drift": r"$\Delta_a$ [in]",
}
display_table(
    story_drift,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Story Drifts",
    label="tab:story_drifts",
)

In [ ]:
# Save dataframes to CSV
shear_walls.to_csv("shear_walls_A.csv")